# LoRA Rank and Batch-Size Experiments with PEFT

This notebook compares several LoRA ranks and per-device batch sizes for FLAN-T5 English-to-German translation. Each run starts from a fresh base model, adds PEFT adapters, trains briefly, and stores comparable metrics.

This code performs a small hyperparameter experiment for LoRA-based English-to-German translation using FLAN-T5 Base and WMT16.

It compares:

LoRA ranks: 1, 4, and 16
Batch sizes: 4, 8, and 16

Therefore, it trains:

3 ranks×3 batch sizes=9 models

The best configuration is selected using the lowest validation loss.

Specifies FLAN-T5 Base as the pretrained model.

Compared with FLAN-T5 Small, the Base model:

Has more parameters.
Usually offers greater modelling capacity.
Requires considerably more GPU memory.
Takes longer to train.

## 1. Install dependencies

In [ ]:
# Remove optional packages that commonly conflict with current
# PyTorch, Transformers, and PEFT installations in Colab.
!pip uninstall -y torchvision torchao

# Install the text-model and LoRA dependencies.
# Do not reinstall torch because Colab already provides it.
!pip install -q -U \
    transformers \
    datasets \
    peft \
    accelerate \
    evaluate \
    sacrebleu \
    sentencepiece

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 9.3 MB/s eta 0:00:00


## 2. Check the environment

In [ ]:
import os
import random
import numpy as np
import torch
import transformers
import datasets
import peft

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
Transformers: 5.14.1
Datasets: 5.0.1
PEFT: 0.20.0
CUDA available: True


## 3. Load and preprocess a shared WMT16 subset

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

raw_dataset = load_dataset( #Downloads the German–English portion of the WMT16 translation dataset.
    "wmt/wmt16",
    "de-en"
)

print(raw_dataset)

validation_split = (
    "validation"
    if "validation" in raw_dataset
    else "test"
)
#The variable validation_split is not used later because the subsequent code directly specifies: split=f"validation[:{VALIDATION_SIZE}]" A more consistent version would use: split=f"{validation_split}[:{VALIDATION_SIZE}]"
print("Validation split:", validation_split)
print(raw_dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 4548885
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2169
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2999
    })
})
Validation split: validation
{'translation': {'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}}


In [ ]:
from datasets import load_dataset

TRAIN_SIZE = 20_000
VALIDATION_SIZE = 1_000

train_raw = load_dataset(
    "wmt/wmt16",
    "de-en",
    split=f"train[:{TRAIN_SIZE}]"
)

validation_raw = load_dataset(
    "wmt/wmt16",
    "de-en",
    split=f"validation[:{VALIDATION_SIZE}]"
)

print(train_raw)
print(validation_raw)
print(train_raw[0])
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128


def preprocess_data(examples): #Defines a function that processes a batch of translation examples.
    inputs = [
        f"translate English to German: {item['en']}"
        for item in examples["translation"]
    ] #Extracts each English sentence and adds a FLAN-T5 instruction.

    targets = [ #Extracts the corresponding German sentences as target outputs.
        item["de"]
        for item in examples["translation"]
    ]

    return tokenizer( #Tokenizes both English inputs and German labels.
        inputs,
        text_target=targets,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )


train_dataset = train_raw.map( #Applies preprocessing to the training subset.
    preprocess_data,
    batched=True,
    remove_columns=train_raw.column_names
)

validation_dataset = validation_raw.map( #Applies the same preprocessing to the validation subset.
    preprocess_data,
    batched=True,
    remove_columns=validation_raw.column_names
)

print(train_dataset.column_names)

Dataset({
    features: ['translation'],
    num_rows: 20000
})
Dataset({
    features: ['translation'],
    num_rows: 1000
})
{'translation': {'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}}


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

['input_ids', 'attention_mask', 'labels']


## 4. Run the experiments

The original notebook used batch sizes 8, 64, and 128. Such large per-device batches commonly exceed Colab GPU memory for FLAN-T5 Base, so this version uses 4, 8, and 16. A larger effective batch can be obtained later with gradient accumulation.

In [ ]:
import gc
import time
import pandas as pd
# gc: Releases Python objects between experiments.
# time: Measures how long each experiment takes.
# pandas: Stores and compares experimental results.
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
) #Imports the model, collator, trainer, and training-configuration classes.
from peft import LoraConfig, TaskType, get_peft_model #Imports the LoRA-related PEFT components.

RANKS = [1, 4, 16]
BATCH_SIZES = [4, 8, 16]
'''
Defines the values that will be tested.

LoRA rank

The rank controls the size of the low-rank matrices used to represent model-weight updates.

r=1: Very few trainable parameters.
r=4: Moderate adapter capacity.
r=16: More trainable parameters and greater adaptation capacity.

A larger rank does not guarantee better validation performance.

Batch size

Batch size controls the number of examples processed per device before one optimizer update.

Smaller batches require less GPU memory.
Larger batches may improve throughput.
Batch size 16 may exceed available memory for FLAN-T5 Base.
'''
EPOCHS = 1
LEARNING_RATE = 5e-4

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported() #Uses BF16 if a CUDA GPU is available and supports it.
USE_FP16 = torch.cuda.is_available() and not USE_BF16 #Uses FP16 if CUDA is available but BF16 is unsupported.
'''
Therefore:

Hardware	            BF16	FP16
Modern  compatible GPU	Yes	    No
Older CUDA GPU	        No	    Yes
CPU	                    No	    No
'''
results = []

for rank in RANKS: #Running all rank and batch-size combinations
    for batch_size in BATCH_SIZES:
        print("\n" + "=" * 80)
        print(f"LoRA rank={rank}, per-device batch size={batch_size}")

        base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME) #Loads a fresh FLAN-T5 Base model for every configuration.
        #This is important because each experiment must begin from the same pretrained weights. Otherwise, one configuration might continue training from the preceding experiment.
        #The disadvantage is that repeatedly loading the model adds some runtime.
        lora_config = LoraConfig( #Creates a LoRA configuration using the current rank from the loop. Sets LoRA alpha to twice the rank.
            task_type=TaskType.SEQ_2_SEQ_LM,
            inference_mode=False,
            r=rank,
            lora_alpha=max(2 * rank, 2),
            lora_dropout=0.05, #Applies 5% dropout to the LoRA path during training.
            target_modules=["q", "v"], #Adds LoRA adapters to the query and value projections in the attention layers.
            bias="none", #Does not train the original bias parameters.
        )

        #The effective LoRA scaling is:

        # lora_alpha/r

        # For the three ranks:

        # Rank	Alpha	Scaling
        # 1	    2	    2
        # 4	    8	    2
        # 16	32	    2

        # Keeping this ratio constant makes the rank comparison more controlled.

        experiment_model = get_peft_model(base_model, lora_config) #Wraps the fresh base model with LoRA adapters. The original FLAN-T5 parameters are frozen, while the newly inserted LoRA parameters are trainable.

        trainable_parameters = sum(
            parameter.numel()
            for parameter in experiment_model.parameters()
            if parameter.requires_grad
        ) #Counts only the trainable LoRA parameters.
        total_parameters = sum(parameter.numel() for parameter in experiment_model.parameters()) #Counts all parameters in the LoRA-wrapped model.
        experiment_model.print_trainable_parameters()

        # Prints a PEFT summary containing:

        # Trainable parameter count.
        # Total parameter count.
        # Trainable percentage.

        # As the rank increases, the number of trainable parameters also increases approximately linearly.

        collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=experiment_model,
            padding=True,
            label_pad_token_id=-100,
            return_tensors="pt",
        )
        # Creates the sequence-to-sequence data collator.

        # padding=True: Pads each batch to its longest sequence.
        # label_pad_token_id=-100: Ensures target padding is ignored by the loss.
        # return_tensors="pt": Produces PyTorch tensors.
        # model=experiment_model: Allows decoder inputs to be prepared properly.

        args = Seq2SeqTrainingArguments( #Defining the settings for one experiment
            output_dir=f"./chapter4_lora_experiments/rank_{rank}_batch_{batch_size}",
            num_train_epochs=EPOCHS,
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            eval_strategy="epoch",
            save_strategy="no",
            logging_strategy="epoch",
            predict_with_generate=False,
            fp16=USE_FP16,
            bf16=USE_BF16,
            report_to="none",
        )

        experiment_trainer = Seq2SeqTrainer( #Creates a trainer for the current experiment.
            model=experiment_model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=validation_dataset,
            data_collator=collator,
            processing_class=tokenizer,
        )
        # Each trainer receives:

        # A newly initialized LoRA model.
        # The current training settings.
        # The same training and validation data.
        # The same data collator.
        # The same tokenizer.
        start_time = time.time() #Records the starting time and assumes the run will complete successfully.
        status = "completed"
        error_message = ""

        try: #Begins a protected section so that one failed configuration does not stop the remaining experiments.
            train_output = experiment_trainer.train() #Trains the current LoRA configuration for one epoch.
            eval_output = experiment_trainer.evaluate()
            #Evaluates it on the validation subset. Because evaluation also runs automatically at the end of the epoch, this call repeats validation once.
            #It ensures that an evaluation result is directly available but increases runtime slightly.
            training_loss = float(train_output.training_loss)
            eval_loss = float(eval_output["eval_loss"])
            runtime_seconds = time.time() - start_time
        except RuntimeError as error: #Catches runtime errors, especially GPU out-of-memory errors. Records that the experiment failed and stores the error message. Uses NaN to represent unavailable loss values.
            # Preserve the remaining experiment runs when one configuration
            # exceeds the available GPU memory.
            status = "failed"
            error_message = str(error)
            training_loss = np.nan
            eval_loss = np.nan
            runtime_seconds = time.time() - start_time
            print("Run failed:", error_message)

        results.append({ #Adds a dictionary describing the current experiment to the results list.
            "rank": rank,
            "batch_size": batch_size,
            "trainable_parameters": trainable_parameters,
            "total_parameters": total_parameters,
            "trainable_percent": 100 * trainable_parameters / total_parameters, #Trainable percentage=(trainable parameters/total parameters)​×100
            "train_loss": training_loss,
            "eval_loss": eval_loss,
            "runtime_seconds": runtime_seconds,
            "status": status,
            "error": error_message,
        })

        # Stores performance and runtime information.Stores whether the experiment completed and any associated error.

        del experiment_trainer, experiment_model, base_model, collator #Deletes references to the current model, trainer, and collator.
        gc.collect() #Requests Python garbage collection to release unused objects.
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        # Releases unused cached CUDA memory. This is especially important because nine FLAN-T5 Base configurations are trained sequentially.

results_df = pd.DataFrame(results)
results_df


LoRA rank=1, per-device batch size=4


model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


...

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


trainable params: 1,769,472 || all params: 249,347,328 || trainable%: 0.7096


Epoch,Training Loss,Validation Loss
1,2.486653,2.347560


Training Loss,Validation Loss,Epoch
2.486653,2.347560,1


,rank,batch_size,trainable_parameters,total_parameters,trainable_percent,train_loss,eval_loss,runtime_seconds,status,error
0,1,4,110592,247688448,0.044650,2.504127,2.336172,939.936775,completed,
1,1,8,110592,247688448,0.044650,2.512347,2.339675,473.637195,completed,
2,1,16,110592,247688448,0.044650,2.523689,2.341833,235.459447,completed,
3,4,4,442368,248020224,0.178360,2.485962,2.338461,956.529595,completed,
4,4,8,442368,248020224,0.178360,2.495018,2.341977,477.326202,completed,
5,4,16,442368,248020224,0.178360,2.507660,2.347330,240.873954,completed,
6,16,4,1769472,249347328,0.709641,2.463537,2.335451,961.357306,completed,
7,16,8,1769472,249347328,0.709641,2.472904,2.341242,481.357718,completed,
8,16,16,1769472,249347328,0.709641,2.486653,2.347560,248.004937,completed,


mportant experimental limitation

The rank comparison is reasonable, but the batch-size comparison is not fully controlled.

With 20,000 training examples:


Batch size	Approximate optimizer steps per epoch

4	        5,000

8	        2,500

16	        1,250


Therefore, the configurations receive different numbers of parameter updates. Batch size changes not only GPU utilization but also the optimization process.

A lower validation loss may be caused by:

The batch size itself.
A different number of optimizer updates.
Different gradient noise.
Interaction with the fixed learning rate.

For a cleaner comparison, use gradient accumulation to maintain the same effective batch size, or compare ranks using one fixed batch size first.

## 5. Compare the completed runs

In [ ]:
completed = results_df[results_df["status"] == "completed"].copy()

if completed.empty:
    print("No experiment completed. Reduce BATCH_SIZES or TRAIN_SAMPLES and run again.")
else:
    display(
        completed.sort_values("eval_loss")[[
            "rank",
            "batch_size",
            "trainable_parameters",
            "trainable_percent",
            "train_loss",
            "eval_loss",
            "runtime_seconds",
        ]]
    )

    best_row = completed.loc[completed["eval_loss"].idxmin()]
    print("\nBest configuration by evaluation loss:")
    print(best_row)

,rank,batch_size,trainable_parameters,trainable_percent,train_loss,eval_loss,runtime_seconds
6,16,4,1769472,0.709641,2.463537,2.335451,961.357306
0,1,4,110592,0.044650,2.504127,2.336172,939.936775
3,4,4,442368,0.178360,2.485962,2.338461,956.529595
1,1,8,110592,0.044650,2.512347,2.339675,473.637195
7,16,8,1769472,0.709641,2.472904,2.341242,481.357718
2,1,16,110592,0.044650,2.523689,2.341833,235.459447
4,4,8,442368,0.178360,2.495018,2.341977,477.326202
5,4,16,442368,0.178360,2.507660,2.347330,240.873954
8,16,16,1769472,0.709641,2.486653,2.347560,248.004937



Best configuration by evaluation loss:
rank                            16
batch_size                       4
trainable_parameters       1769472
total_parameters         249347328
trainable_percent         0.709641
train_loss                2.463537
eval_loss                 2.335451
runtime_seconds         961.357306
status                   completed
error                             
Name: 6, dtype: object


## 6. Save experiment results

In [ ]:
RESULTS_PATH = "chapter4_lora_experiment_results.csv"
results_df.to_csv(RESULTS_PATH, index=False)
print("Saved:", RESULTS_PATH)

Saved: chapter4_lora_experiment_results.csv
